# Fine-tuning com Correções Reais — Classificador de Soja

Ajusta o modelo (`soja_model_final.keras`) com as fotos reais coletadas no app
(dataset `Guguinhaxd/soja-correction`).

## Como a progressão é medida (HOLDOUT)

Um conjunto fixo de ~100 fotos reais (`holdout/` no dataset HF) **nunca entra no treino**.
Toda rodada avalia o modelo nesse mesmo conjunto, antes e depois — é a régua que
prova se o modelo melhorou de verdade entre versões.

⚠️ **Pré-requisito:** rodar `model/build_holdout.py` UMA vez antes da primeira rodada
(e com `--rebuild` quando quiser incorporar fotos novas ao holdout).

## Safety gate (promoção do modelo)

O modelo só substitui o de produção se **AMBAS** as condições valerem:
1. Teste original ≥ 90% do baseline (perda máxima de 10% relativa no laboratório)
2. Holdout real **melhor** que o baseline (melhora absoluta no seu domínio)

Rejeitados são salvos como `soja_model_rejected_<timestamp>.keras` para análise.

### Pré-requisitos
1. `soja_model_final.keras` no Drive (gerado pelo `train.ipynb`)
2. Holdout criado (`build_holdout.py`)
3. GPU ligada: `Ambiente de execução → Alterar tipo → T4 GPU`

Rode as células **em ordem, sem pular**.

In [ ]:
# Célula 1 — GPU, seeds, imports e login no Hugging Face
!pip install -q huggingface_hub

import os, json, glob
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from huggingface_hub import snapshot_download, HfApi

# Reprodutibilidade: fixa numpy + python random + tensorflow de uma vez
tf.keras.utils.set_random_seed(42)

print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {gpus}')
assert len(gpus) > 0, 'Sem GPU! Ambiente de execução → Alterar tipo → T4'

# Token: https://huggingface.co/settings/tokens (leitura no dataset, escrita no Space)
from getpass import getpass
HF_TOKEN = getpass('Cole seu token do Hugging Face: ')

In [ ]:
# Célula 2 — Montar Drive, caminhos e classes
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT     = '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)'
TRAIN_DIR        = f'{DATASET_ROOT}/train'
TEST_DIR         = f'{DATASET_ROOT}/test'
SAVE_DIR         = '/content/drive/MyDrive'
RESULTS_DIR      = f'{SAVE_DIR}/results'
BASE_MODEL_PATH  = f'{SAVE_DIR}/soja_model_final.keras'
CORRECTIONS_REPO = 'Guguinhaxd/soja-correction'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Mesma ordem do train.ipynb — NÃO mudar (índice = saída do modelo)
CLASS_NAMES = [
    'Broken soybeans',
    'Immature soybeans',
    'Intact soybeans',
    'Skin-damaged soybeans',
    'Spotted soybeans',
]
SHORT = {
    'Broken soybeans':       'broken',
    'Immature soybeans':     'immature',
    'Intact soybeans':       'intact',
    'Skin-damaged soybeans': 'skin-damaged',
    'Spotted soybeans':      'spotted',
}
NUM_CLASSES = len(CLASS_NAMES)

assert os.path.isfile(BASE_MODEL_PATH), f'Modelo base não encontrado: {BASE_MODEL_PATH}'
assert os.path.isdir(TRAIN_DIR), f'Dataset original não encontrado: {TRAIN_DIR}'
print('OK — modelo base e dataset original encontrados.')

In [ ]:
# Célula 3 — Baixar correções + holdout do Hugging Face
CORR_DIR = snapshot_download(
    repo_id=CORRECTIONS_REPO,
    repo_type='dataset',
    token=HF_TOKEN,
    local_dir='/content/correcoes',
)
print(f'Dataset baixado em: {CORR_DIR}\n')

# Holdout é OBRIGATÓRIO — sem régua fixa, melhoria não é mensurável
MANIFEST_PATH = os.path.join(CORR_DIR, 'holdout_manifest.json')
assert os.path.isfile(MANIFEST_PATH), (
    'holdout_manifest.json não encontrado no dataset!\n'
    'Rode primeiro: python model/build_holdout.py (uma única vez)'
)
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
holdout_names = {n for names in manifest['files'].values() for n in names}
print(f'Holdout: {sum(len(v) for v in manifest["files"].values())} imagens (seed={manifest["seed"]})')

def list_images(folder):
    return sorted(
        glob.glob(os.path.join(folder, '*.jpg')) +
        glob.glob(os.path.join(folder, '*.jpeg')) +
        glob.glob(os.path.join(folder, '*.png'))
    )

corr_counts, hold_counts = {}, {}
print(f'\n{"classe":>14} | treino | holdout')
for cls in CLASS_NAMES:
    train_files = [
        f for f in list_images(os.path.join(CORR_DIR, cls))
        if os.path.basename(f) not in holdout_names   # cinto E suspensório
    ]
    hold_files = list_images(os.path.join(CORR_DIR, 'holdout', cls))
    corr_counts[cls] = train_files
    hold_counts[cls] = hold_files
    print(f'{SHORT[cls]:>14} | {len(train_files):>6} | {len(hold_files):>7}')

# Verificação do critério de aceite: NENHUM arquivo do holdout no treino
train_names = {os.path.basename(f) for files in corr_counts.values() for f in files}
leaked = train_names & holdout_names
assert not leaked, f'VAZAMENTO! Arquivos do holdout no treino: {leaked}'
print('\n✅ Verificado: zero interseção entre treino e holdout.')

total_corr = sum(len(v) for v in corr_counts.values())
total_hold = sum(len(v) for v in hold_counts.values())
assert total_corr >= 5,  'Poucas correções de treino! Colete mais fotos pelo app.'
assert total_hold >= 5,  'Holdout vazio/minúsculo — rode build_holdout.py.'
print(f'Total — treino: {total_corr} | holdout: {total_hold}')

In [ ]:
# Célula 4 — Recortar cada grão (MESMA lógica do app.py)
# O modelo classifica o grão RECORTADO. Treino, holdout e inferência precisam
# do mesmo enquadramento. Idempotente: imagem já recortada é mantida.
import cv2
from PIL import Image as PILImage

def crop_single_grain(arr):
    h, w    = arr.shape[:2]
    gray    = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    thresh  = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    thresh  = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=1)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return arr
    largest = max(contours, key=cv2.contourArea)
    if not (0.02 < cv2.contourArea(largest) / (h * w) < 0.95):
        return arr
    x, y, bw, bh = cv2.boundingRect(largest)
    pad = max(15, int(min(bw, bh) * 0.12))
    return arr[max(0, y-pad):min(h, y+bh+pad), max(0, x-pad):min(w, x+bw+pad)]

def crop_folder(counts, out_root):
    out_counts = {}
    for cls in CLASS_NAMES:
        out_dir = os.path.join(out_root, cls)
        os.makedirs(out_dir, exist_ok=True)
        saved = []
        for fp in counts[cls]:
            try:
                arr  = np.array(PILImage.open(fp).convert('RGB'))
                crop = crop_single_grain(arr)
                dst  = os.path.join(out_dir, os.path.basename(fp))
                PILImage.fromarray(crop).save(dst, format='JPEG', quality=95)
                saved.append(dst)
            except Exception as e:
                print(f'  pulei {os.path.basename(fp)}: {e}')
        out_counts[cls] = saved
    return out_counts

corr_counts = crop_folder(corr_counts, '/content/correcoes_cropped')
hold_counts = crop_folder(hold_counts, '/content/holdout_cropped')
print('Recorte concluído — treino e holdout no mesmo enquadramento do modelo.')

In [ ]:
# Célula 5 — Split interno treino/validação ESTRATIFICADO + dataset do holdout
# O split interno (80/20) serve só para EarlyStopping/checkpoint durante o treino.
# A régua oficial de progresso é o HOLDOUT, que não participa de nada disso.
rng = np.random.default_rng(42)
ct_paths, ct_labels, cv_paths, cv_labels = [], [], [], []

for idx, cls in enumerate(CLASS_NAMES):
    files = list(corr_counts[cls])
    if not files:
        print(f'{SHORT[cls]:>14}: SEM correções — classe fica só com o dataset original')
        continue
    rng.shuffle(files)
    n_val = max(1, int(len(files) * 0.2))
    cv_paths  += files[:n_val];  cv_labels += [idx] * n_val
    ct_paths  += files[n_val:];  ct_labels += [idx] * (len(files) - n_val)
    print(f'{SHORT[cls]:>14}: {len(files)-n_val:>3} treino | {n_val:>2} val interna')

ct_paths  = np.array(ct_paths);  ct_labels = np.array(ct_labels, dtype=np.int32)
cv_paths  = np.array(cv_paths);  cv_labels = np.array(cv_labels, dtype=np.int32)

# Holdout (avaliação apenas — nunca treina, nunca early-stoppa)
ho_paths, ho_labels = [], []
for idx, cls in enumerate(CLASS_NAMES):
    ho_paths  += list(hold_counts[cls])
    ho_labels += [idx] * len(hold_counts[cls])
ho_paths  = np.array(ho_paths);  ho_labels = np.array(ho_labels, dtype=np.int32)

assert len(ct_paths) > 0, 'Nenhuma correção para treino!'
print(f'\nTreino: {len(ct_paths)} | val interna: {len(cv_paths)} | HOLDOUT: {len(ho_paths)}')

In [ ]:
# Célula 6 — Pipelines tf.data com PESO e MIX adaptativos por volume
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

def corr_weight(n_correcoes: int) -> float:
    """Peso na loss por exemplo de correção. Com poucos dados o peso alto só
    acelera a memorização (overfitting visto na rodada 1: 57 fotos, treino 89%
    vs val 64%). O peso cresce junto com o volume, quando é seguro usá-lo."""
    if n_correcoes < 150:  return 1.0
    if n_correcoes < 300:  return 1.5
    return 2.0

def mix_ratio(n_correcoes: int) -> float:
    """Fração de cada lote vinda das correções (resto = dataset original)."""
    if n_correcoes < 150:  return 0.4
    if n_correcoes < 300:  return 0.55
    return 0.7

n_corr      = len(ct_paths)
CORR_WEIGHT = corr_weight(n_corr)
MIX         = mix_ratio(n_corr)
ORIG_WEIGHT = 1.0
print(f'n_correções={n_corr}  →  CORR_WEIGHT={CORR_WEIGHT}  |  mix={MIX:.0%} correções / {1-MIX:.0%} original')

# Augmentation FORTE nas correções (fotos reais do setup do usuário)
aug_strong = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(height_factor=0.12, width_factor=0.12),
    layers.RandomBrightness(0.25),
    layers.RandomContrast(0.25),
], name='aug_strong')

# Augmentation LEVE no dataset original (replay anti-esquecimento)
aug_light = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomBrightness(0.05),
], name='aug_light')

def load_image(path, label):
    img   = tf.io.read_file(path)
    img   = tf.io.decode_image(img, channels=3, expand_animations=False)
    img   = tf.image.resize(img, IMG_SIZE)
    img   = tf.cast(img, tf.float32)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

orig_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, class_names=CLASS_NAMES, image_size=IMG_SIZE,
    batch_size=None, label_mode='categorical', shuffle=True, seed=42,
).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)

corr_raw = (
    tf.data.Dataset.from_tensor_slices((ct_paths, ct_labels))
    .shuffle(len(ct_paths), seed=42, reshuffle_each_iteration=True)
    .map(load_image, num_parallel_calls=AUTOTUNE)
)

# .batch() → .map(aug) → .unbatch(): camadas Keras de augmentation pedem 4D.
# O 3º elemento é o sample_weight, aplicado automaticamente na loss pelo fit().
corr_stream = (
    corr_raw.repeat()
    .batch(BATCH_SIZE)
    .map(lambda x, y: (aug_strong(x, training=True), y,
                       tf.ones([tf.shape(x)[0]]) * CORR_WEIGHT), num_parallel_calls=AUTOTUNE)
    .unbatch()
)
orig_stream = (
    orig_raw.repeat()
    .batch(BATCH_SIZE)
    .map(lambda x, y: (aug_light(x, training=True), y,
                       tf.ones([tf.shape(x)[0]]) * ORIG_WEIGHT), num_parallel_calls=AUTOTUNE)
    .unbatch()
)

mixed = (
    tf.data.Dataset.sample_from_datasets(
        [corr_stream, orig_stream], weights=[MIX, 1.0 - MIX], seed=42
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

def eval_ds(paths, labels):
    return (
        tf.data.Dataset.from_tensor_slices((paths, labels))
        .map(load_image, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )

corr_val   = eval_ds(cv_paths, cv_labels)   # val interna (EarlyStopping)
holdout_ds = eval_ds(ho_paths, ho_labels)   # régua oficial

STEPS_PER_EPOCH = max(60, (len(ct_paths) * 8) // max(1, int(BATCH_SIZE * MIX)))
print(f'Steps por época: {STEPS_PER_EPOCH}')

In [ ]:
# Célula 7 — Carregar modelo, medir BASELINES e descongelar camadas
model = keras.models.load_model(BASE_MODEL_PATH)
model.compile(loss='categorical_crossentropy', metrics=['accuracy'])
print('Modelo carregado.')

# Teste original (laboratório) — para o gate de esquecimento
test_ds = (
    tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, class_names=CLASS_NAMES, image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False,
    )
    .map(lambda x, y: (tf.cast(x, tf.float32), y))
    .prefetch(AUTOTUNE)
)

# BASELINES do modelo ATUAL (antes de qualquer treino) — referência do gate
_, baseline_original = model.evaluate(test_ds,    verbose=0)
_, baseline_holdout  = model.evaluate(holdout_ds, verbose=0)
print(f'BASELINE — teste original (lab):  {baseline_original:.1%}')
print(f'BASELINE — holdout (fotos reais): {baseline_holdout:.1%}')

# Descongela últimas 30 camadas; BatchNorm fica congelada (estatísticas ImageNet)
base_model = next((l for l in model.layers if isinstance(l, keras.Model)), None)
if base_model is not None:
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    trainable = sum(1 for l in base_model.layers if l.trainable)
    print(f'Camadas treináveis na base: {trainable}/{len(base_model.layers)}')
else:
    print('Base aninhada não encontrada — treinando só a cabeça.')

In [ ]:
# Célula 8 — Fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),   # LR 100× menor que o treino original
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        patience=6, restore_best_weights=True,
        monitor='val_accuracy', verbose=1),
    keras.callbacks.ModelCheckpoint(
        f'{SAVE_DIR}/soja_finetuned_best.keras',
        save_best_only=True, monitor='val_accuracy', verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        patience=3, factor=0.5, monitor='val_loss',
        min_lr=1e-7, verbose=1),
]

print(f'=== Fine-tuning — {MIX:.0%} correções (peso {CORR_WEIGHT}) / {1-MIX:.0%} original ===')
history = model.fit(
    mixed,
    validation_data=corr_val,
    epochs=20,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks,
)

In [ ]:
# Célula 9 — Avaliação no HOLDOUT + matriz de confusão + histórico em JSON
from sklearn.metrics import confusion_matrix, classification_report

_, new_holdout  = model.evaluate(holdout_ds, verbose=0)
_, new_original = model.evaluate(test_ds,    verbose=0)

print('═' * 58)
print(f'HOLDOUT (fotos reais) — antes: {baseline_holdout:.1%}  →  depois: {new_holdout:.1%}'
      f'  ({(new_holdout-baseline_holdout)*100:+.1f} pp)')
print(f'TESTE ORIGINAL (lab)  — antes: {baseline_original:.1%}  →  depois: {new_original:.1%}'
      f'  ({(new_original-baseline_original)*100:+.1f} pp)')
print('═' * 58)

# Por classe, no holdout
y_pred = np.argmax(model.predict(holdout_ds, verbose=0), axis=1)
y_true = ho_labels
short_names = [SHORT[c] for c in CLASS_NAMES]
print('\n=== Relatório por classe (HOLDOUT) ===')
print(classification_report(y_true, y_pred, labels=range(NUM_CLASSES),
                            target_names=short_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
per_class_acc = {
    short_names[i]: (float(cm[i, i] / cm[i].sum()) if cm[i].sum() else None)
    for i in range(NUM_CLASSES)
}

# Histórico comparável entre rodadas
ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
eval_record = {
    'timestamp':          ts,
    'n_train_corrections': int(n_corr),
    'corr_weight':        CORR_WEIGHT,
    'mix_corrections':    MIX,
    'holdout_size':       int(len(ho_paths)),
    'baseline_holdout':   round(float(baseline_holdout), 4),
    'new_holdout':        round(float(new_holdout), 4),
    'baseline_original':  round(float(baseline_original), 4),
    'new_original':       round(float(new_original), 4),
    'per_class_holdout':  per_class_acc,
    'confusion_matrix':   cm.tolist(),
}
eval_path = f'{RESULTS_DIR}/eval_{ts}.json'
with open(eval_path, 'w') as f:
    json.dump(eval_record, f, ensure_ascii=False, indent=2)
print(f'Resultados salvos: {eval_path}')

# Gráficos
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(history.history['accuracy'],     label='Treino (misto)')
axes[0].plot(history.history['val_accuracy'], label='Val interna')
axes[0].set_title('Acurácia'); axes[0].set_xlabel('Época'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history.history['loss'],     label='Treino')
axes[1].plot(history.history['val_loss'], label='Val interna')
axes[1].set_title('Loss'); axes[1].set_xlabel('Época'); axes[1].legend(); axes[1].grid(alpha=0.3)
im = axes[2].imshow(cm, cmap='Blues')
axes[2].set_xticks(range(NUM_CLASSES)); axes[2].set_xticklabels(short_names, rotation=45, ha='right')
axes[2].set_yticks(range(NUM_CLASSES)); axes[2].set_yticklabels(short_names)
axes[2].set_xlabel('Predito'); axes[2].set_ylabel('Real')
axes[2].set_title('Matriz de confusão (HOLDOUT)')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        axes[2].text(j, i, cm[i, j], ha='center', va='center',
                     color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/finetune_{ts}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Célula 10 — SAFETY GATE: promove ou rejeita o modelo
# Promoção exige AMBAS as condições:
#   1. Teste original ≥ 90% do baseline (perda máx. de 10% RELATIVA no lab)
#   2. Holdout real estritamente MELHOR que o baseline
cond_1 = new_original >= 0.90 * baseline_original
cond_2 = new_holdout  >  baseline_holdout

print('═══ SAFETY GATE ═══')
print(f'  Teste original:  {new_original:.1%}  (mínimo exigido: {0.90*baseline_original:.1%})'
      f'  →  {"✅ PASSOU" if cond_1 else "❌ REPROVOU"}')
print(f'  Holdout real:    {new_holdout:.1%}  (baseline a superar: {baseline_holdout:.1%})'
      f'  →  {"✅ PASSOU" if cond_2 else "❌ REPROVOU"}')

if cond_1 and cond_2:
    model.save(f'{SAVE_DIR}/soja_model_final.keras')
    print('\n✅ PROMOVIDO — soja_model_final.keras atualizado, pronto para o Space.')
    PROMOTED = True
else:
    rejected_path = f'{SAVE_DIR}/soja_model_rejected_{ts}.keras'
    model.save(rejected_path)
    print(f'\n❌ REJEITADO — modelo de produção mantido.')
    print(f'   Modelo rejeitado salvo para análise: {rejected_path}')
    if not cond_2:
        print('   → Holdout não melhorou: colete mais fotos das classes fracas')
        print('     (veja a matriz de confusão da Célula 9).')
    if not cond_1:
        print('   → Esqueceu demais o lab: o mix/peso adaptativos já deveriam prevenir isso;')
        print('     verifique se as correções têm rótulos certos (fotos na pasta errada?).')
    PROMOTED = False

In [ ]:
# Célula 11 (OPCIONAL) — Subir modelo + eval pro HF Space via API
# SÓ RODE SE A CÉLULA 10 MOSTROU ✅ PROMOVIDO.
assert PROMOTED, 'Modelo foi REJEITADO pelo gate — não suba para o Space!'
SPACE_REPO = 'Guguinhaxd/soja-inspection'

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=f'{SAVE_DIR}/soja_model_final.keras',
    path_in_repo='soja_model_final.keras',
    repo_id=SPACE_REPO,
    repo_type='space',
    commit_message=f'fine-tune: holdout {baseline_holdout:.0%}→{new_holdout:.0%}',
)
# Sobe também o eval para a aba Status do app mostrar a última avaliação
api.upload_file(
    path_or_fileobj=eval_path,
    path_in_repo=f'results/eval_{ts}.json',
    repo_id=SPACE_REPO,
    repo_type='space',
    commit_message=f'eval: {ts}',
)
print('✅ Modelo + avaliação enviados pro Space. Rebuild automático em ~2 min.')